In [136]:

from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV
from imblearn.under_sampling import RandomUnderSampler 
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import confusion_matrix
from sklearn.model_selection import cross_val_score
import numpy as np
from sklearn.metrics import accuracy_score
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from collections import defaultdict
import scipy
rseed = 6740
np.random.seed(rseed)

In [137]:
class loadData:
    def __init__(self,randomstate=6740):
        self.test_data_fashion = pd.read_csv('data/fashion-mnist_test.csv')                              
        self.train_data_fashion = pd.read_csv('data/fashion-mnist_train.csv')                            
        self.data_digits = scipy.io.loadmat('data/mnist_10digits.mat')                                   
                                                                                                        
        self.train_data_x_fashion = self.train_data_fashion.iloc[:, 1:]                                  
        self.train_data_y_fashion = self.train_data_fashion.iloc[:, 0]                                   
        self.test_data_x_fashion  = self.test_data_fashion.iloc[:, 1:]                                   
        self.test_data_y_fashion  = self.test_data_fashion.iloc[:, 0]                                    

        self.train_data_x_digits = self.data_digits['xtrain']                                            
        self.train_data_y_digits = self.data_digits['ytrain']                                          
        self.test_data_x_digits  = self.data_digits['xtest']                                             
        self.test_data_y_digits  = self.data_digits['ytest']                                             
                                                                                                        
        scaler_fashion = MinMaxScaler()                                                                  
        scaler_digits  = MinMaxScaler()                                                                  
                                                                                                        
        self.fashion_train_data_x_scaled = scaler_fashion.fit_transform(self.train_data_x_fashion)       
        self.fashion_test_data_x_scaled  = scaler_fashion.transform(self.test_data_x_fashion)            
                                                                                                        
        self.digits_train_data_x_scaled = scaler_digits.fit_transform(self.train_data_x_digits)          
        self.digits_test_data_x_scaled  = scaler_digits.transform(self.test_data_x_digits)

   


    

In [138]:
data_sets = loadData()

In [139]:
def logisitc_regression(data_x,data_y):
    log_reg_model = LogisticRegression(random_state=6740,max_iter=2000)
    log_reg_model.fit(data_x,data_y)
    return log_reg_model

def knn(data_x,data_y,k):
    knn_model = KNeighborsClassifier(n_neighbors=k)
    knn_model.fit(data_x,data_y)
    return knn_model

def mlp(data_x,data_y):
    mlp_model = MLPClassifier(hidden_layer_sizes=(20,10),random_state=6740,max_iter=50)
    mlp_model.fit(data_x, data_y)
    return mlp_model

def svm(data_x,data_y,kernel=False):
    rand_downsample = np.random.choice(data_x.shape[0], 5000, replace=False) 
    data_x_downsampled = data_x[rand_downsample]
    data_y_downsampled = data_y[rand_downsample]
    if kernel ==False:
        svm_model = SVC(kernel='linear',random_state=6740)
    else:
        svm_model = SVC(kernel='rbf',random_state=6740)
    
    svm_model.fit(data_x_downsampled,data_y_downsampled)

    return svm_model

    
    




In [140]:
fashion_model_dict = {}

In [141]:
fashion_model_dict = {}
digit_model_dict = {}
fashion_model_dict['logistic_regression']=logisitc_regression(data_sets.fashion_train_data_x_scaled,data_sets.train_data_y_fashion)
fashion_model_dict['mlp']=mlp(data_sets.fashion_train_data_x_scaled,data_sets.train_data_y_fashion)
fashion_model_dict['knn']=knn(data_sets.fashion_train_data_x_scaled,data_sets.train_data_y_fashion,4)
fashion_model_dict['svm']=svm(data_sets.fashion_train_data_x_scaled,data_sets.train_data_y_fashion)
fashion_model_dict['rbf']=svm(data_sets.fashion_train_data_x_scaled,data_sets.train_data_y_fashion,kernel=True)

digit_model_dict['logistic_regression']=logisitc_regression(data_sets.digits_train_data_x_scaled,data_sets.train_data_y_digits[0])
digit_model_dict['mlp']=mlp(data_sets.digits_train_data_x_scaled,data_sets.train_data_y_digits[0])
digit_model_dict['knn']=knn(data_sets.digits_train_data_x_scaled,data_sets.train_data_y_digits[0],4)
digit_model_dict['svm']=svm(data_sets.digits_train_data_x_scaled,data_sets.train_data_y_digits[0])
digit_model_dict['rbf']=svm(data_sets.digits_train_data_x_scaled,data_sets.train_data_y_digits[0],kernel=True)

/Users/willbrennan/Desktop/Coding/school_repo/school_code/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(
/Users/willbrennan/Desktop/Coding/school_repo/school_code/.venv/lib/python3.13/site-packages/sklearn/neural_network/_multilayer_perceptron.py:785: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


In [142]:
fashion_confusion_matrix = {}
digit_confusion_matrix = {}
for fashion,digits in zip(fashion_model_dict.items(),digit_model_dict.items()):

    fashion_model_predictions = fashion[1].predict(data_sets.fashion_test_data_x_scaled)
    digit_model_predictions = digits[1].predict(data_sets.digits_test_data_x_scaled)

    fashion_confusion_matrix[fashion[0]]=confusion_matrix(data_sets.test_data_y_fashion,fashion_model_predictions)
    digit_confusion_matrix[digits[0]]=confusion_matrix(data_sets.test_data_y_digits[0],digit_model_predictions)

In [145]:
fashion_confusion_matrix['mlp']

array([[843,   4,  20,  52,   0,   0,  65,   1,  14,   1],
       [  2, 979,   1,  14,   0,   1,   3,   0,   0,   0],
       [ 15,   2, 812,  14,  72,   0,  80,   0,   5,   0],
       [ 23,  21,   9, 915,  12,   3,  16,   0,   1,   0],
       [  0,   4,  86,  42, 776,   0,  90,   0,   2,   0],
       [  0,   0,   0,   1,   0, 923,   1,  48,   8,  19],
       [154,   5, 114,  47,  55,   0, 603,   0,  22,   0],
       [  0,   0,   0,   0,   0,  26,   0, 947,   0,  27],
       [  8,   0,   6,   5,   2,   4,  17,   2, 956,   0],
       [  0,   0,   0,   0,   0,  17,   0,  62,   1, 920]])

In [146]:
digit_confusion_matrix['mlp']

array([[ 944,    0,    4,    3,    4,    8,    5,    5,    4,    3],
       [   0, 1105,    4,    1,    0,    2,    2,    3,   18,    0],
       [   0,    2,  992,   10,    2,    3,    5,    9,    7,    2],
       [   1,    0,   11,  951,    0,   22,    1,    9,   13,    2],
       [   1,    0,    6,    2,  918,    2,    5,   10,    9,   29],
       [   2,    0,    0,   10,    0,  863,    6,    3,    7,    1],
       [   8,    2,    2,    1,    4,   20,  912,    1,    8,    0],
       [   0,    2,   10,    8,    0,    4,    0,  989,    9,    6],
       [   5,    4,    0,   16,    4,   19,    4,    7,  906,    9],
       [   6,    4,    0,    8,    7,    4,    1,    7,   13,  959]])

In [144]:
# param_grid = {'n_neighbors': range(1, 21)}

# knn_model = KNeighborsClassifier()


# grid_search = GridSearchCV(knn_model, param_grid, cv=5)
# grid_search.fit(data_sets.fashion_train_data_x_scaled,data_sets.train_data_y_fashion)

# grid_search.best_params_['n_neighbors']

